<a href="https://colab.research.google.com/github/ssprajapati2021/Hybrid-RAG-Fine-Tuning/blob/main/notebook/Solution_V1_RAG_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 5: Solution V1 — RAG Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` — Created by Notebooks 3+4
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `v1_metrics.csv` — Per-row Baseline vs V1 scores _(Evidence for comparative analysis)_

---

### **Task 3.3: Evaluate Solution V1**

> This task is split into five measurements (3.3.1–3.3.5). Run the shared setup cell below first (it loads the model, ChromaDB, and test data), then work through each measurement.

**── Shared setup ──**
Load the base model, reload ChromaDB (same embedding model as NB4), and load `df_test.csv` + `outputs.json`. Define helper functions `generate_baseline()` and `generate_naive_rag()` here so every subtask below can reuse them.

In [2]:
# Installing Required Packages
!pip install -q langchain-huggingface sentence-transformers
!pip install -q langchain-community
!pip install -q chromadb langchain-chroma
!pip install -q -U bitsandbytes>=0.46.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 122.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [1]:
# Mount Google Drive to access the artifacts
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
#Import Libraries
import json
import os
import pandas as pd
import torch

from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig)

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Define Paths
artifact_path = "/content/drive/MyDrive/corporate_policies"

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load the base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

# Load the Embedding Model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Load the ChromaDB
vector_db = Chroma(
    persist_directory=os.path.join(artifact_path, "chroma_db"),
    embedding_function=embedding_model
)

# Load the df_test.csv
df_test = pd.read_csv(
    os.path.join(artifact_path, "df_test.csv")
)

df_test.head()

# Load the outputs.json
with open(os.path.join(artifact_path, "outputs.json"), "r") as f:
    outputs = json.load(f)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
# Define Helper Function generate_baseline

def generate_baseline(query):

  messages = [
      {
          "role" : "system",
          "content": "You are a helpful customer support assistant."
      },
      {
          "role": "user",
          "content":query
      }
  ]

  prompt = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
  )

  inputs = tokenizer(
      prompt,
      return_tensors="pt"
  ).to(model.device)

  outputs = model.generate(
      **inputs,
      max_new_tokens=120,
      do_sample=False,
      temperature=None,
      top_p=None
  )
  generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

  return tokenizer.decode(
      generated_tokens,
      skip_special_tokens=True
  )

In [9]:
# Define Helper Function generate_naive_rag

def generate_naive_rag(query):

  docs = vector_db.similarity_search(
      query,
      k=1
  )

  context = docs[0].page_content

  messages = [
        {
            "role": "system",
            "content": f"""Answer strictly using this SOP.
              SOP:
              {context}

              If the answer is not present in the SOP, say you don't know."""
        },
        {
            "role": "user",
            "content": query
        }
    ]

  prompt = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
  )

  inputs = tokenizer(
      prompt,
      return_tensors="pt"
  ).to(model.device)

  outputs = model.generate(
      **inputs,
      max_new_tokens=120,
      do_sample=False,
      temperature=None,
      top_p=None
  )

  generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

  return tokenizer.decode(
      generated_tokens,
      skip_special_tokens=True
  )

In [10]:
print(generate_baseline(outputs["test_query"]))

print("\n",generate_naive_rag(outputs["test_query"]))

I'm sorry to hear that your package is taking longer than expected. There could be several reasons for this delay, such as traffic or road conditions, weather-related delays, or issues with the carrier's system. I recommend checking the status of your package on the carrier's website or by calling their customer service number. They can provide more information and help resolve any issues that may be causing the delay. If the issue persists, it might be best to contact the carrier directly to see if they can expedite delivery.

 I'm sorry to hear that your package is running behind schedule. To better understand the situation, could you please confirm the order identifier and the original estimated delivery date when you placed the order? This will help us determine if there were any issues with the shipping process or if there was a change in the shipping company's service levels.


#### **3.3.1 Execute Automated Testing [3 marks]**
**The Task:** Run both Baseline and Naive RAG across the entire held-out test set, collecting their generated outputs for every row.

**Hints & Tips:**
* Loop over `df_test` rows; for each query call both `generate_baseline()` and `generate_naive_rag()`.
* Store raw outputs in a list of dicts so the later measurements can score them.
* This is the most time-consuming cell — if constrained, `df_test.sample(50)` is acceptable.

**Learner Inference:** Automated testing across the full set gives statistically meaningful results, not a single cherry-picked query.

In [11]:
# Creating Evaluation dataframe with sample 50 on df_test
evaluation_df = df_test.sample(50, random_state=42)

results = []

for _, row in evaluation_df.iterrows():

    query = row["instruction"]

    baseline_output = generate_baseline(query)
    rag_output = generate_naive_rag(query)

    results.append({
        "instruction": query,
        "intent": row["intent"],
        "category": row["category"],
        "baseline_output": baseline_output,
        "naive_rag_output": rag_output
    })

results_df = pd.DataFrame(results)

results_df.head()

,instruction,intent,category,baseline_output,naive_rag_output
0,I cannot afford order {{Order Number}},cancel_order,ORDER,I'm sorry to hear that you're unable to afford...,"If you cannot afford the order, please check y..."
1,I need assistance to enter another shipping ad...,set_up_shipping_address,SHIPPING,"Sure, I'd be happy to help! To change your shi...","To proceed, please provide the new shipping ad..."
2,need assistance changing the details on my pro...,edit_account,ACCOUNT,"Sure, I'd be happy to help you change your pro...",To assist you with changing your profile detai...
3,I do not know what I have to do to speak with ...,contact_human_agent,CONTACT,If you're having trouble speaking with an oper...,You can try calling the number on your phone's...
4,I have problems entering a different shipping ...,set_up_shipping_address,SHIPPING,If you're having trouble entering a different ...,If you're having issues entering a different s...


In [12]:
results_df.to_csv(
    os.path.join(artifact_path, "evaluation_results.csv"),
    index=False
)

#### **3.3.2 Measure Format Adherence [2 marks]**
**The Task:** Validate the syntactic correctness of the generated outputs and report the adherence rate.

**Hints & Tips:**
* For the baseline/RAG free-text responses, "format adherence" means the output is well-formed and non-empty (the strict JSON check applies mainly to the fine-tuned router in NB7).
* Report the percentage of outputs that parsed/validated successfully.

**Learner Inference:** Format adherence tells you how often the system produces usable output before you even check correctness.

In [13]:
# Defining a validate_format function to validate format
def validate_format(outputs, is_json_schema=False):
    valid_count = 0
    total_count = len(outputs)

    for out in outputs:
        if is_json_schema:
            # Check JSON parseability
            try:
                json.loads(out)
                valid_count += 1
            except (json.JSONDecodeError, TypeError):
                continue
        else:
            # Check non-empty / valid string
            if isinstance(out, str) and len(out.strip()) > 0:
                valid_count += 1

    adherence_rate = (valid_count / total_count) * 100 if total_count > 0 else 0.0
    return valid_count, total_count, adherence_rate



#### **3.3.3 Measure Execution Success (ROUGE/BLEU) [2 marks]**
**The Task:** Evaluate semantic similarity of each output against SOP-grounded references using ROUGE-1, ROUGE-L, and BLEU.

**Hints & Tips:**
* Use SOP-grounded references — retrieve the correct SOP per test row so policy-specific language is rewarded.
* Generic references falsely reward vague baseline answers — avoid them.
* `rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)` and `sentence_bleu` with `SmoothingFunction().method1`.

**Learner Inference:** ROUGE/BLEU measure how close the output is to a correct, policy-grounded answer.

In [ ]:
# YOUR CODE HERE


#### **3.3.4 Measure Output Consistency [1 mark]**
**The Task:** Evaluate deterministic behaviour by running the same query multiple times under `do_sample=False` and confirming identical outputs.

**Hints & Tips:**
* Run the same query 3 times; assert all outputs are identical.
* With `do_sample=False, temperature=None`, greedy decoding should be fully deterministic.

**Learner Inference:** Deterministic inference means your evaluation is reproducible — the same input always gives the same output.

In [ ]:
# YOUR CODE HERE


#### **3.3.5 Measure Hallucination Frequency [2 marks]**
**The Task:** Evaluate how often outputs contain unsupported claims, invalid references, missing functionality, or policy violations.

**Hints & Tips:**
* Compare outputs against the retrieved SOP — flag any specific claim (dates, numbers, policies) not supported by the context.
* Report hallucination frequency as a percentage for both Baseline and Naive RAG.

**Learner Inference:** This quantifies the core problem RAG is meant to solve — grounding responses to reduce fabrication.

In [ ]:
# YOUR CODE HERE


### **Task 3.4: Analyse Retrieval Impact**

#### **3.4.1 Compare Baseline and Solution V1 [4 marks]**
**The Task:** Quantify the impact of retrieval by comparing aggregate scores across Functional Correctness, Consistency, and Hallucination Frequency, with percentage changes.

**Hints & Tips:**
* Build a summary table: Baseline vs Naive RAG for each metric.
* Compute improvement percentages: `(rag - base) / base * 100`.
* Document WHERE retrieval helps and where it doesn't — both motivate Stage 4.

**Learner Inference:** This isolates retrieval's independent contribution before fine-tuning enters the picture.

In [ ]:
# YOUR CODE HERE


---
## Save Artifacts

In [ ]:
# YOUR CODE HERE


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 6.**

- [ ] ChromaDB reloaded from `./chroma_db/`
- [ ] **3.3.1** Automated testing run across full test set
- [ ] **3.3.2** Format adherence measured
- [ ] **3.3.3** ROUGE/BLEU computed with SOP-grounded references
- [ ] **3.3.4** Output consistency (determinism) verified
- [ ] **3.3.5** Hallucination frequency quantified
- [ ] **3.4.1** Retrieval impact quantified with improvement %
- [ ] **`v1_metrics.csv` saved** ← _Evidence for comparative analysis_

**If any item is unchecked, fix it before moving on.**